# Week 2 Day 3 assignment: a handler can reject a payout

In the lab, a payout over $1,000 pauses the graph until a handler approves it. The only answer was yes.

Here the handler's answer carries `approved` and a `note`. When a handler rejects a payout, the note goes back to the agent as a message, and the agent looks at the claim again. Once a handler has rejected a payout on a claim, any payout the agent comes back with goes to the handler again, whatever the amount.

We try it on CLM-1011, a lost suitcase on the way to a wedding, where the payout is easy to get wrong. The handler paid $1,150.

Run it from the `undercover` folder's `.venv` kernel, with `OPENROUTER_API_KEY` and `DATABASE_URL` in `.env`, as for the lab.

## Setup

The same setup as the lab: OpenRouter, Supabase in the `lab` schema, UnderCover's data and the payout tool from Week 2 Day 1.

In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from psycopg.rows import dict_row
from pydantic import BaseModel, Field

ROOT = Path("../..")
sys.path.insert(0, str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout

In [ ]:
load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")

connection = psycopg.connect(DATABASE_URL, autocommit=True, prepare_threshold=0, row_factory=dict_row, options="-c search_path=lab")
connection.execute("CREATE SCHEMA IF NOT EXISTS lab")

luna = ChatOpenAI(model="openai/gpt-6-luna", base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)
print("Connected")

In [ ]:
DATA = ROOT / "data"


def load(name):
    """Read one of UnderCover's JSON files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = {claim["claim_id"]: claim for claim in load("claims.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
APPROVAL_LIMIT = load("company.json")["payout_approval_limit"]

The instructions, the prompt, the payout tool and the agent and decide nodes are the lab's, unchanged. The state has one new field, `rejected`, which records that a handler has turned down a payout on this claim.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use the payout tool to work out the payout on an approved claim. When you have decided, give your decision, the payout and your reasons.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)


class ClaimState(TypedDict):
    """What the graph keeps for one claim."""

    messages: Annotated[list, add_messages]
    claim_id: str
    decision: str
    payout: float
    rationale: str
    approved_by: str
    rejected: bool


class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")


model_with_tools = luna.bind_tools([calculate_payout])
decider = luna.with_structured_output(ClaimDecision)
system_message = SystemMessage(f"{INSTRUCTIONS}\n{policy_wording}\n")


def agent(state: ClaimState):
    """Call the model with the conversation so far."""
    return {"messages": [model_with_tools.invoke([system_message, *state["messages"]])]}


def decide(state: ClaimState):
    """Turn the agent's work into a decision."""
    result = decider.invoke([system_message, *state["messages"], HumanMessage("Record your decision on this claim.")])
    return {"decision": result.decision, "payout": result.payout, "rationale": result.rationale}

## The change: approve or reject

The approval node now picks the next node itself. It returns a `Command`, which updates the state and says where to go:

- **Approved:** record who approved it and go to `pay`, as before.
- **Rejected:** add the handler's note to the conversation as a message and go back to `agent`. The agent sees its own work, then the note, and can call the payout tool again.

The return type, `Command[Literal["pay", "agent"]]`, tells LangGraph which nodes the approval node can go to, so the graph needs no fixed edge out of it and the picture shows both ways.

As in the lab, LangGraph runs the node again from the top when it resumes, so the first message prints twice.

In [ ]:
def approval(state: ClaimState) -> Command[Literal["pay", "agent"]]:
    """Wait for a handler to approve or reject a payout over the limit."""
    print(f"{state['claim_id']} needs a handler to approve ${state['payout']:,.2f}")
    answer = interrupt({"claim_id": state["claim_id"], "payout": state["payout"], "rationale": state["rationale"]})

    if answer["approved"]:
        return Command(goto="pay", update={"approved_by": answer["handler"]})

    print(f"{answer['handler']} rejected ${state['payout']:,.2f}: {answer['note']}")
    note = HumanMessage(
        f"A handler, {answer['handler']}, rejected your payout of ${state['payout']:,.2f}.\n\n"
        f"Their note: {answer['note']}\n\n"
        "Look at the claim again with the note in mind and give your decision, the payout and your reasons."
    )
    return Command(goto="agent", update={"messages": [note], "rejected": True})

The pay node and the checkpointer are the lab's. Two things change in the edges:

- The fixed edge from `approval` to `pay` has gone, because the approval node chooses for itself.
- `after_decide` sends a payout to the handler when it's over the limit **or** when a handler has already rejected a payout on this claim. Without that, a corrected payout under $1,000 would be paid without the handler seeing it. My first run did exactly that: the agent came back with $895, which was wrong, and the graph paid it.

In [ ]:
connection.execute("""
CREATE TABLE IF NOT EXISTS payments (
    claim_id TEXT PRIMARY KEY,
    amount NUMERIC NOT NULL,
    approved_by TEXT,
    paid_at TIMESTAMPTZ DEFAULT now()
)
""")


def pay(state: ClaimState):
    """Pay the claim, unless it has been paid already."""
    cursor = connection.execute(
        "INSERT INTO payments (claim_id, amount, approved_by) VALUES (%s, %s, %s) ON CONFLICT (claim_id) DO NOTHING",
        (state["claim_id"], state["payout"], state.get("approved_by")),
    )
    if cursor.rowcount:
        print(f"Paid {state['claim_id']} ${state['payout']:,.2f}")
    else:
        print(f"{state['claim_id']} was already paid")


def after_agent(state: ClaimState):
    """Run the tools the model asked for, or move on to the decision."""
    if state["messages"][-1].tool_calls:
        return "tools"
    return "decide"


def after_decide(state: ClaimState):
    """Pay an approval, after a handler's approval if it's over the limit or a handler has rejected a payout before."""
    if state["decision"] != "approve":
        return END
    if state["payout"] > APPROVAL_LIMIT or state.get("rejected"):
        return "approval"
    return "pay"


checkpointer = PostgresSaver(connection)
checkpointer.setup()

builder = StateGraph(ClaimState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode([calculate_payout]))
builder.add_node("decide", decide)
builder.add_node("approval", approval)
builder.add_node("pay", pay)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", after_agent, ["tools", "decide"])
builder.add_edge("tools", "agent")
builder.add_conditional_edges("decide", after_decide, ["approval", "pay", END])
builder.add_edge("pay", END)

graph = builder.compile(checkpointer=checkpointer)
display(Image(graph.get_graph().draw_mermaid_png()))

## CLM-1011: the best man's suitcase

The agent decides the claim, and the graph pauses if the payout is over $1,000.

In [ ]:
def start_fresh(claim_id):
    """Delete a claim's saved thread and its payment."""
    checkpointer.delete_thread(claim_id)
    connection.execute("DELETE FROM payments WHERE claim_id = %s", (claim_id,))


start_fresh("CLM-1011")
config = {"configurable": {"thread_id": "CLM-1011"}}
claim = {"messages": [HumanMessage(make_user_prompt(claims["CLM-1011"]))], "claim_id": "CLM-1011"}

result = graph.invoke(claim, config, durability="sync")
print(f"\n{result['decision']} ${result['payout']:,.2f}: {result['rationale']}")

## The handler checks the payout

The handler, Dana Whitfield, works the claim out by hand and gets $1,150:

| Line | Counts at |
|---|---|
| Suitcase | $260 |
| Tuxedo jacket and trousers, sold as a two piece suit: one item, at the $300 limit | $300 |
| Dress shoes, a pair: one item | $230 |
| Camera body with its lens: one item, at the $300 limit | $300 |
| Four dress shirts: four items at $85 | $340 |
| Silver cufflinks, a pair: one item | $120 |
| Less the carrier's settlement | −$350 |
| Less the Premium deductible | −$50 |
| **Payout** | **$1,150** |

When the agent's payout matches, the handler approves it. When it doesn't, the handler rejects it with a note that points to the rules the agent needs, without giving it the answer. This goes on for up to three rounds.

In [ ]:
HANDLER = "Dana Whitfield"
HANDLER_PAYOUT = 1150.0
NOTE = (
    "Check the payout against section 5 and the per-item limit on Premium. "
    "A suit counts as one item, and so does a camera with its lens, so look at how the receipts sold them. "
    "Several shirts are separate items. Take off the carrier's settlement and the deductible."
)


def handler_review(config, rounds=3):
    """Approve the payout if it matches the handler's, or reject it with a note, until the claim moves on."""
    for round_number in range(1, rounds + 1):
        state = graph.get_state(config)
        if state.next != ("approval",):
            break
        payout = state.values["payout"]
        approved = abs(payout - HANDLER_PAYOUT) < 0.01
        print(f"--- Round {round_number}: the agent says ${payout:,.2f}, the handler says ${HANDLER_PAYOUT:,.2f}")
        answer = {"approved": approved, "handler": HANDLER, "note": "" if approved else NOTE}
        result = graph.invoke(Command(resume=answer), config, durability="sync")
    return result


result = handler_review(config)
if graph.get_state(config).next == ("approval",):
    print(f"\nStill waiting for the handler after three rounds, at ${result['payout']:,.2f}")
else:
    print(f"\nFinal: {result['decision']} ${result['payout']:,.2f}, approved by {result.get('approved_by')}")
print(f"Rationale: {result['rationale']}")

### What the agent saw

The conversation in the saved state shows the handler's note arriving as a message and the agent's reply to it.

In [ ]:
for message in graph.get_state(config).values["messages"][1:]:
    if message.type == "human":
        print(f"HANDLER NOTE:\n{message.content}\n")
    elif message.type == "ai" and message.tool_calls:
        for call in message.tool_calls:
            lines = ", ".join(f"{line['description']} ${line['amount']:,.0f}" + (f" x{line['quantity']}" if line.get("quantity", 1) > 1 else "") for line in call["args"]["lines"])
            print(f"AGENT CALLS {call['name']}: {lines}; paid by others ${call['args'].get('paid_by_others', 0):,.0f}\n")
    elif message.type == "tool":
        print(f"TOOL: payout ${json.loads(message.content)['payout']:,.2f}\n")
    elif message.type == "ai":
        print(f"AGENT:\n{message.text}\n")

### The payment and the checkpoints

One payment, at the amount the handler approved. The checkpoints show the way the claim took through the graph, newest first.

In [ ]:
pd.DataFrame(connection.execute("SELECT * FROM payments WHERE claim_id = %s", ("CLM-1011",)).fetchall())

In [ ]:
for snapshot in graph.get_state_history(config):
    print(snapshot.metadata["step"], snapshot.next)

## Takeaways

- **The note becomes part of the conversation.** It's saved in the checkpoint with everything else, so a rejection survives a crash just as an approval does, and the agent's second attempt is on the record next to its first.
- **`Command` lets a node choose where to go.** A conditional edge is still the clearer choice when the next step depends only on the state; here it depends on the handler's answer, which only the node has.
- **A rejection has to stick.** The approval limit decides when a handler sees a payout the first time. After a rejection the handler has the last word, or a corrected payout under the limit slips through unchecked, right or wrong.
- **A note can fix one mistake and expose another.** The agent often gets the sets right after the note and then gets something else wrong, such as passing the four shirts as $85 when the payout tool wants the total for the line. That's why the handler checks every round.